In [1]:
#  -- PART 0: Import Relevant Modules 
import pandas as pd
import numpy as np
import torch 

from datasets import Dataset
from transformers import RobertaTokenizer, RobertaForSequenceClassification, TrainingArguments, Trainer, set_seed 
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, classification_report

: 

In [ ]:
# to maintain consistent results
set_seed(100)

# -- PART 1: Read Files 
training_df = pd.read_csv("cleaned_training_data.csv")
test_df = pd.read_csv("cleaned_test_data.csv")

,Topic,ID,Segment,IdeaUnit,label,NoteText
0,ComputerScience,6260226,1,declarative knowledge is a factual statement,1,basics of computer science declarative knowled...
1,ComputerScience,6260226,1,imperative knowledge is solving a problem or a...,1,basics of computer science declarative knowled...
2,ComputerScience,6260226,1,algorithms are instructions with steps to comp...,1,basics of computer science declarative knowled...


In [ ]:
# Topic = class in question 
# ID = student id 
# segment = section of student notes 
# IdeaUnit = idea that the notes should contain 
# label = 1 if accurate representation, 0 otherwise 
# NoteText = student's note (what's evaluated against IdeaUnit, needs to match each of those key components)

In [ ]:
# -- PART 2: Pre-Processing Steps

# -- (a) create minimal df
training_df = training_df[['IdeaUnit', 'NoteText', 'label']]
test_df = test_df[['IdeaUnit', 'NoteText', 'label']]

,IdeaUnit,NoteText,label
0,declarative knowledge is a factual statement,basics of computer science declarative knowled...,1
1,imperative knowledge is solving a problem or a...,basics of computer science declarative knowled...,1
2,algorithms are instructions with steps to comp...,basics of computer science declarative knowled...,1


In [ ]:
# -- (b) convert datasets to HuggingFace dataset
hf_training_df = Dataset.from_pandas(training_df)
hf_test_df = Dataset.from_pandas(test_df)

In [ ]:
# -- PART 3: Train BERT Model (compare IdeaUnit and NoteText against each other and provide label 1)

# (a) load model and tokenizer 
model_name = 'roberta-base' 

tokenizer = RobertaTokenizer.from_pretrained(model_name)
model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels = 2)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# -- (a.1) verifying truncation is valid for tokenization 
def get_length_of_token(row): 
    token = tokenizer(row['IdeaUnit'], row['NoteText'], truncation=False)['input_ids']
    return len(token)

In [ ]:
train_note_length = []
for i in range(len(hf_training_df)): 
    length = get_length_of_token(hf_training_df[i])
    train_note_length.append(length)

test_note_length = []
for j in range(len(hf_test_df)):  
    length = get_length_of_token(hf_test_df[j])
    test_note_length.append(length)    

In [ ]:
print("TRAINING DATASET")
print("- mean:", np.mean(train_note_length))
print("- min" , min(train_note_length))
print("- max" , max(train_note_length))
print(">512 sanity check", np.sum(np.array(train_note_length) > 512))

print("")

print("TEST DATASET")
print("- mean:", np.mean(test_note_length))
print("- min" , min(test_note_length))
print("- max" , max(test_note_length))
print(">512 sanity check", np.sum(np.array(test_note_length) > 512))

TRAINING DATASET
- mean: 102.66666666666667
- min 41
- max 169
>512 sanity check 0

TEST DATASET
- mean: 80.65194648425712
- min 7
- max 318
>512 sanity check 0


In [ ]:
# -- (b) encode IdeaUnit and NoteText (use cased because capitalization does matter here)
def tokenize_input(examples): 
    return tokenizer(
        examples['IdeaUnit'],
        examples['NoteText'], 
        max_length = 512, 
        padding = 'max_length', 
        truncation = True
    )

In [ ]:
# map training and test dataset to preprocessing function 
tokenized_train_df = hf_training_df.map(tokenize_input, batched = True)
tokenized_train_df.set_format('torch')

tokenized_test_df = hf_test_df.map(tokenize_input, batched = True)
tokenized_test_df.set_format('torch')

tokenized_train_df

Map:   0%|          | 0/255 [00:00<?, ? examples/s]

Map:   0%|          | 0/9941 [00:00<?, ? examples/s]

Dataset({
    features: ['IdeaUnit', 'NoteText', 'label', 'input_ids', 'attention_mask'],
    num_rows: 255
})

In [ ]:
# -- PART 4: Train BERT Model (compare IdeaUnit and NoteText against each other and provide label 1)

# assign training arguments (hyperparameters)
training_args = TrainingArguments(
    output_dir = 'training_arguments', 
    learning_rate = 2e-05,
    num_train_epochs = 20,
    weight_decay = 0.01, 
    seed = 100, 
    data_seed = 100
    #warmup_ratio = 0.1
)

# train model 
trainer = Trainer(
    model = model, 
    args = training_args, 
    train_dataset = tokenized_train_df
)

trainer.train()

Step,Training Loss
500,0.293000


TrainOutput(global_step=640, training_loss=0.22910701183136553, metrics={'train_runtime': 107.7491, 'train_samples_per_second': 47.332, 'train_steps_per_second': 5.94, 'total_flos': 1341866382336000.0, 'train_loss': 0.22910701183136553, 'epoch': 20.0})

In [ ]:
# -- PART 5: Predict on Test Data

# get predictions 
predicted_class = trainer.predict(tokenized_test_df)

# convert raw scores to binary classification (choose larger)
predicted_class = np.argmax(predicted_class.predictions, axis = 1)

# add predicted values to original dataset
test_df['predicted_class'] = predicted_class 

In [ ]:
# -- PART 6: Evaluate Results (Accuracy)

print("Accuracy:", accuracy_score(test_df['label'], test_df['predicted_class']))
print("F1 Score:", f1_score(test_df['label'], test_df['predicted_class'], average = 'macro'))
print("Precision Score:", precision_score(test_df['label'], test_df['predicted_class'], average = 'macro'))
print("Recall Score:", recall_score(test_df['label'], test_df['predicted_class'], average = 'macro'))

print("")

print(classification_report(test_df['label'], test_df['predicted_class'], target_names = ['Not Covered', 'Covered']))

Accuracy: 0.7575696609998994
F1 Score: 0.7372420678895196
Precision Score: 0.737328723537781
Recall Score: 0.73715605724723

              precision    recall  f1-score   support

 Not Covered       0.81      0.81      0.81      6350
     Covered       0.66      0.66      0.66      3591

    accuracy                           0.76      9941
   macro avg       0.74      0.74      0.74      9941
weighted avg       0.76      0.76      0.76      9941

